In [2]:
import time
from photonic_testing import *
MODBUS_PORT='/dev/tty.usbserial-B003T6PZ'
# MODBUS_PORT='/dev/tty.usbserial-B003T6RF'
LASER_ADDRESSES = {"1028": 5,
                   "1270": 6,
                   "yj1430": 3,
                   "hk1430": 4,
                   "1510": 1,
                   "2330": 2}

LASER_DRIVER_SERIALS = {"1028": 8229,
                   "1270": 8228,
                   "yj1430": 8227,
                   "hk1430": 8222,
                   "1510": 8225,
                   "2330": 8226}

### Overview

See `ait/photonic_testing.py` for `Laser` and `LaserProperties` along with the specific limits of the individual laser diodes.


### Low-level access

In the event that direct device control is needed.

See `ait/maiman_modbus/utils/utils.py` for python constants of register names and `ait/maiman_modbus/config/modbus_config.yaml` for the register addresses.

Look at methods on `ModbusDevice` (`ait/maiman_modbus/device/modbus_device.py`) for functions.


```python
from maiman_modbus.communication import ModbusCommunication
import maiman_modbus.utils as maiman_regs
from maiman_modbus.device.modbus_device_model import ModbusDeviceModel
from maiman_modbus.config import DeviceConfig
from maiman_modbus.device.modbus_device import ModbusDevice

d = ModbusDevice(port=MODBUS_PORT, slave_address=modbus_address)
d.comm.send_command(d.model.get_register(STATE_OF_TEC_COMMAND), MODBUS_START_TEC_COMMAND_VALUE)

print(d.comm.receive_response(d.model.get_register(STATE_OF_TEC_COMMAND)))
```


## Initialize all the diodes

Running this cell will create the `lasers` dictionary with a `Laser` for each laser.

In [3]:
name = ('yj1430', )
names = tuple(LASER_ADDRESSES.keys())
lasers = {}
for name in names:
    l = Laser(name, address=LASER_ADDRESSES[name], MODBUS_PORT=MODBUS_PORT)
    serial = l.device.get_serial_number()
    print(f'🆔 Serial number: {serial}')
    assert l.device.get_serial_number()==LASER_DRIVER_SERIALS[name], 'BAD BUS CONFIG, do not continue'
    lasers[name] = l
    print('')


for name in names:
    serial = lasers[name].device.get_serial_number()
    assert serial==LASER_DRIVER_SERIALS[name], 'BAD BUS CONFIG, do not continue'

Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
🆔 Serial number: 8229

Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
🆔 Serial number: 8228

Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
🆔 Serial number: 8227

Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
🆔 Serial number: 8222

Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
🆔 Serial number: 8225

Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
🆔 Serial number: 8226



### Start the TECs

In [4]:
for name in names:
    lasers[name].program_drive_limits()
for name in names:
    lasers[name].disable_interlock_and_cool()

Programming Limits for Maiman (S/N: 8229)...
Programming Limits for Maiman (S/N: 8228)...
Programming Limits for Maiman (S/N: 8227)...
Programming Limits for Maiman (S/N: 8222)...
Programming Limits for Maiman (S/N: 8225)...
Programming Limits for Maiman (S/N: 8226)...


Look at the status of one of them. It seems that the TEC status isn't polling well, but the temp changes.

In [7]:
l = lasers['1270']
l.status()
tec_current = l.device.comm.receive_response(l.device.model.get_register('tec_current_measured'))
tec_voltage = l.device.comm.receive_response(l.device.model.get_register('tec_voltage'))
print(f'\nTEC Current: {tec_current} Voltage: {tec_voltage} V')

Laser Properties Name: 1270
Raw ID from register: 0x1113
Device ID: 4371
Serial Number: 8228
State: 0xf7
 Operation started: True
 Current Set Internal: True
 Enable Internal: True
 External NTC Denied: True
 Interlock Denied: True
Current: 0.0
Current Min: 0.0
Current Max: 70.0
Max Current Limit: 250.0
Protection Threshold: 109.2
Voltage: 0.8
Frequency: 0.0
Duration: 1.0
Raw PCB temperature (signed): 0
PCB Temp: 0.0
Current Set Calibration: 100.0
TEC PID: (100, 1000, 0)
TEC Voltage: 0.0
TEC Current Limit: 1.0
TEC Current: 0.0
TEC Temperature Setpoint: 25.0
TEC Temperature: 24.99
TEC NTC Coefficient: 3988.0
TEC State: 0x16
 TEC started: True
 TEC Set Internal: True
 TEC Enable Internal: True
Interlock State: 0x2
 Interlock: True
 LD Overcurrent: False
 LD Overheat: False
 External NTC Interlock: False
 TEC Error: False
 TEC Self-heat: False
TEC Current: 0 Voltage: 0 V


### Turn on a laser

This sets a percentage between the maximum current and the threshold current.

In [9]:
lasers['2330'].set_current_as_percent(.5)

Programming Limits for Maiman (S/N: 8226)...
Setting current to 96.225 mA 


### Make sure it is all off.

In [10]:
for name in names:
    lasers[name].shutdown()